# 🌍 Tourist Attraction Aspect Scorer
## **Fine-Tuning RoBERTa for Multi-Label Regression**

---

### 📝 **Project Overview**
In this project, we develop a specialized AI model designed to "understand" tourist attractions through the eyes of travelers. While standard sentiment analysis only classifies a review as positive or negative, our model performs **Multi-Label Regression**.

It analyzes raw text descriptions and predicts **10 distinct aspect scores** (on a scale of 0-10), including:
* **Vibe:** Romance, Family, Adventure, Relaxation.
* **Practicality:** Cost, Service, Accessibility.
* **Experience:** Nature, Culture, Food.

---

### 🤖 **The Model: Why RoBERTa?**
We utilize **RoBERTa** (*Robustly Optimized BERT approach*), a powerful Transformer-based architecture. RoBERTa excels at capturing the nuances of human language, such as sarcasm, context-dependent meanings, and subtle descriptions.

By fine-tuning this model on tourism-specific datasets, we transform it from a general language model into a **domain expert** for the travel industry.

---

### 🚀 **The Technical Challenge**
The core of this project is mapping unstructured text to a **continuous distribution of values**.
* **Multi-Label:** The model predicts 10 scores simultaneously.
* **Regression:** Instead of choosing a category, the model estimates a precise intensity for each aspect.
* **Sequential Fine-Tuning:** We use advanced training techniques to refine the model's accuracy on "edge cases" (extremely high or low scores).

---

### 🛠️ **Workflow Pipeline**
1.  **Data Preparation:** Normalizing labels and structuring the JSON dataset.
2.  **Tokenization:** Converting text into numerical tensors for the Transformer.
3.  **Training:** Running the fine-tuning process using the Hugging Face `Trainer` API.
4.  **Inference:** Using the model to score new, unseen travel itineraries in real-time.

---

## **1. Environment Setup**
Before we begin, we need to install the necessary libraries.
* **Transformers & Accelerate:** To load and train the RoBERTa model.
* **Datasets:** To handle our JSON/CSV data efficiently.
* **Scikit-Learn:** To calculate evaluation metrics like Mean Absolute Error (MAE).

In [24]:
# Install the required libraries
# We use [torch] to ensure all dependencies for PyTorch training are included
!pip install -q transformers[torch] datasets scikit-learn accelerate

import torch
import pandas as pd
import json
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import mean_absolute_error

# Check if GPU is available for training
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


## **2. Evaluation Metrics**
Since we are performing a regression task, we need a way to measure how far the model's predictions are from the actual human-labeled scores.

We use **Mean Absolute Error (MAE)** as our primary metric. However, because our model is trained on normalized labels (scaled between 0 and 1), we must "denormalize" the results back to their original 0-10 range to get a meaningful error score.

For example, an **MAE of 1.0** means that on average, the model's rating is off by just one point on the 0-10 scale.

In [25]:
def compute_metrics(eval_pred):
    """
    the function takes the model's predictions and the true labels,
    scales them back to the original 0-10 range,
    and calculates the mean absolute error (MAE) between the predicted and actual values.
    The MAE is returned in a dictionary format, which can be used by the Trainer to evaluate the model's performance during training.
    :param eval_pred: tuple of (logits, labels) where logits are the model's predictions and labels are the true values.
    :return: a dictionary with the mean absolute error (MAE) between the predicted and actual values, scaled back to the original 0-10 range.
    """
    # The model was trained to predict values in the range of 0-1 (by dividing the original 0-10 scores by 10).
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    # Scale predictions and labels back to the original 0-10 range
    predictions = logits * 10.0
    actuals = labels * 10.0
    # Calculate MAE
    mae = mean_absolute_error(actuals, predictions)
    return {"mae": mae}

## **3. Data Preparation & Normalization**
Deep learning models, especially Transformers, perform better when target values are scaled within a small range. In this section, we convert our raw JSON input into a structured **Hugging Face Dataset** object.

**Key operations performed here:**
* **Feature Extraction:** Separating the raw review text from the aspect scores.
* **Normalization:** Scaling our 0-10 scores down to a **0-1 range** (by dividing by 10). This helps the model's loss function converge more smoothly.
* **Multi-Label Structuring:** Organizing the 10 different categories into a single vector for each review, allowing for simultaneous multi-aspect prediction.

In [26]:
def prepare_data(data_list):
    """
    the function takes a list of dictionaries (each containing a review and its associated distribution of scores)
    and prepares it for training by extracting the text and labels, normalizing the labels to a 0-1 range,
     and creating a Hugging Face Dataset object.
    :param data_list: a list of dictionaries, where each dictionary has a "review" key containing the text of the review and a "distribution" key containing a dictionary of scores for various aspects (e.g., "Romance", "Family", etc.).
     The scores in the "distribution" are expected to be in the range of 0-10, and they will be normalized to a 0-1 range for training the model.
    :return: a Hugging Face Dataset object containing the processed text and labels, ready for training a sequence classification model. The labels are normalized to a 0-1 range by dividing the original scores by 10.
    """
    texts = []
    labels = []
    label_keys = ["Romance", "Family", "Cost", "Nature", "Adventure",
                  "Culture", "Food", "Relaxation", "Service", "Accessibility"]
    # Loop through each item in the input list, extract the review text and the corresponding distribution of scores, normalize the scores to a 0-1 range, and append them to the respective lists.
    for item in data_list:
        texts.append(item["review"])
        dist = [float(item["distribution"][key]) / 10.0 for key in label_keys]
        labels.append(dist)

    # Create the dataset
    dataset = Dataset.from_dict({"text": texts, "labels": labels})
    return dataset


## **4. Tokenization: Converting Text to Tensors**
Transformers do not read raw strings. They process sequences of integers called **Tokens**. In this step, we use the model's specific tokenizer to break down our reviews into a numerical format.

**Key features of this process:**
* **Subword Tokenization:** Handling rare words by breaking them into smaller, meaningful pieces.
* **Padding & Truncation:** Ensuring all input sequences have a fixed length (128 tokens). This is necessary for efficient parallel processing on the GPU.
* **Attention Masks:** Telling the model which parts of the input are actual text and which are just "padding" added to fill the space.

In [ ]:
def tokenize_function(examples, tokenizer):
    """
    the function takes a batch of examples and a tokenizer,
     and applies the tokenizer to the "text" field of the examples.
     The tokenizer will convert the raw text into a format suitable for input into a transformer model,
      such as token IDs, attention masks, etc. The function also applies padding and truncation to ensure that all sequences
       are of the same length (128 tokens in this case) for efficient batch processing during training.
    :param examples: a batch of examples, where each example is expected to have a "text" field containing the review text that needs to be tokenized. The function will process this field using the provided tokenizer.
    :param tokenizer: a tokenizer object from the Hugging Face Transformers library, which will be used to convert the raw text in the "text" field of the examples into token IDs and attention masks. The tokenizer will also handle padding and truncation to ensure that all sequences are of the same length (128 tokens in this case) for efficient batch processing during training.
    :return: a dictionary containing the tokenized outputs, including input IDs, attention masks, and any other relevant fields produced by the tokenizer. The tokenized outputs will be used as input for training a transformer model on the sequence classification task.
    """
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


## **5. Model Orchestration & Training**
In this final stage, we combine all the components into a single training pipeline.

### **The Model: DeBERTa v3 Small**
For this task, we chose **DeBERTa v3 Small**. Despite its compact size (~141M parameters), it is more efficient and powerful than standard BERT or RoBERTa models. It uses an improved attention mechanism (disentangled attention) and a "replaced token detection" objective, making it ideal for high-precision regression tasks on relatively small datasets.

### **The Training Strategy**
* **Task:** Multi-Label Regression (10 output labels).
* **Optimizer:** AdamW with a learning rate of $2 \times 10^{-5}$ for stable weight updates.
* **Epochs:** 5 iterations over the full dataset to ensure deep learning without excessive overfitting.
* **Checkpointing:** We save only the "Best Model" based on the validation loss, ensuring that we keep the version that generalizes best to new, unseen reviews.

In [3]:
def main(raw_json_data):
    """
    the main function orchestrates the entire process of fine-tuning
    a transformer model for sequence classification on a dataset of reviews.
    It starts by loading a pre-trained model and tokenizer,
    prepares the dataset by processing the raw JSON data,
    tokenizes the text data, and then sets up the training arguments
    and trainer to fine-tune the model.
    :param raw_json_data: a list of dictionaries, where each dictionary contains a "review" key with the text of the review and a "distribution" key with a dictionary of scores for various aspects (e.g., "Romance", "Family", etc.). This raw JSON data will be processed to create a dataset suitable for training a transformer model for sequence classification. The function will handle the entire workflow from data preparation to model training.
    :return: None. The function will fine-tune the model and save the best model checkpoint to the specified output directory ("./tourism_model") based on the evaluation metrics computed during training.
    """
    # the model for finetune
    # the model have ~141 Million parameters, and it is a good choice for fine-tuning on a dataset of 10,000 reviews, as it provides a good balance between performance and computational efficiency. The DeBERTa-v3-small model is designed to capture complex language patterns and nuances, making it suitable for tasks like sentiment analysis and regression on review data.
    model_name = "microsoft/deberta-v3-small"
    # get the tokenizer for the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # define the model for sequence classification with 10 labels (for the 10 aspects) and specify that it's a regression problem
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=10,
        problem_type="regression"
    )

    # send the model to the appropriate device (GPU if available, otherwise CPU) and ensure it uses float32 precision for training
    model.to(torch.float32)

    # prepare the dataset by processing the raw JSON data, which includes extracting the review text and normalizing the distribution of scores to a 0-1 range for training
    full_dataset = prepare_data(raw_json_data)

    # split the dataset into training and testing sets (90% for training and 10% for testing) and apply the tokenization function to the text data in a batched manner for efficient processing during training
    tokenized_datasets = full_dataset.train_test_split(test_size=0.1).map(
        lambda x: tokenize_function(x, tokenizer), batched=True
    )

    # set the format of the tokenized datasets to PyTorch tensors, specifying that the "input_ids", "attention_mask", and "labels" fields should be included in the output for training the model
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # define the training arguments for fine-tuning the model, including the output directory for saving the model checkpoints, evaluation and saving strategies, learning rate, batch size, number of training epochs, weight decay, and whether to load the best model at the end of training based on evaluation metrics
    training_args = TrainingArguments(
        output_dir="./tourism_model",
        eval_strategy="epoch",
        save_strategy="epoch",
        fp16=False,
        bf16=False,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        save_total_limit=1,
        weight_decay=0.01,
        load_best_model_at_end=True,
    )

    # define the Trainer object, which will handle the training loop, evaluation, and saving of model checkpoints based on the specified training arguments and the tokenized datasets for training and evaluation. The compute_metrics function will be used to evaluate the model's performance during training by calculating the mean absolute error (MAE) between the predicted and actual values.
    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
    )

    # train the model using the Trainer, which will handle the entire training process, including forward and backward passes, optimization, evaluation, and saving of model checkpoints based on the specified training arguments and evaluation metrics.
    trainer.train()

## **6. Execution: Starting the Fine-Tuning Process**
Now that our pipeline is defined, it is time to load our curated dataset and begin the training process.

**What happens when we run this?**
1. **Data Loading:** We load the `for_full_fineTune.json` file containing our travel reviews and labels.
2. **Preprocessing:** The `main` function will automatically handle the split between training and validation data.
3. **Training Loop:** The model will process the reviews in batches, adjusting its weights to minimize the error in its predictions.
4. **Evaluation:** At the end of each epoch, the model will be tested on unseen data to track its learning progress.

In [19]:
if __name__ == "__main__":
    # Placeholder: Load your 10,000 reviews here
    with open('for_full_fineTune.json', 'r') as f:
        raw_data = json.load(f)
    main(raw_data)
    pass

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

Map:   0%|          | 0/5189 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

## **7. Model Refinement (Sequential Fine-Tuning)**
Sometimes, after the initial training, we identify specific "edge cases" or new datasets where the model needs extra precision. Instead of retraining from scratch, we perform **Sequential Fine-Tuning**.

This function allows us to load our best checkpoint and continue training on a smaller, targeted dataset (like the LLM-labeled extreme cases).

**Key Adjustments for Refined Training:**
* **Lower Learning Rate ($1 \times 10^{-5}$):** We use a very small learning rate to avoid "Catastrophic Forgetting," ensuring the model retains its prior knowledge while slowly adapting to new patterns.
* **Cosine Learning Rate Scheduler:** This creates a smooth decay of the learning rate, helping the model settle into a more precise local minimum.
* **Warmup Steps:** A gradual start that prevents the model from being "shocked" by the new data gradients in the first few steps.
* **Increased Regularization:** Using higher weight decay to prevent the model from overfitting on this smaller subset of data.

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

def second_fine_tune(new_data):
    # Load the previously fine-tuned model from the checkpoint
    # Note: Make sure the path is correct and accessible
    model_path = "/content/local_model"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # Prepare the new dataset and tokenize it
    full_dataset = prepare_data(new_data)
    tokenized_datasets = full_dataset.train_test_split(test_size=0.1).map(
        lambda x: tokenize_function(x, tokenizer), batched=True
    )
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # Define refined training arguments for the second pass
    training_args = TrainingArguments(
        output_dir="./tourism_model_refined",
        eval_strategy="epoch",
        save_strategy="epoch",

        # Hyperparameters for continued training
        learning_rate=1e-5,               # Stable LR to refine weights without overshooting
        lr_scheduler_type="cosine",        # Gradually reduces LR for better convergence
        warmup_steps=100,                 # Starts slow to protect pre-learned weights
        weight_decay=0.1,                 # Increased regularization to prevent overfitting

        per_device_train_batch_size=16,
        num_train_epochs=3,               # Just enough to see the new data patterns

        # Model selection and saving
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", # Crucial for regression tasks
        greater_is_better=False,
        save_total_limit=1,

        # Logging
        logging_steps=10,
        report_to="none"                  # Set to "wandb" if you use it, otherwise "none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics
    )

    print("Starting refined training...")
    trainer.train()

    # Save the final refined model and tokenizer
    trainer.save_model("./tourism_model_final")
    tokenizer.save_pretrained("./tourism_model_final")

In [10]:
if __name__ == "__main__":
    with open('train_ready_data.json', 'r') as f:
        raw_data = json.load(f)
    second_fine_tune(raw_data)
    pass

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/2289 [00:00<?, ? examples/s]

Map:   0%|          | 0/255 [00:00<?, ? examples/s]

Starting refined training...


Epoch,Training Loss,Validation Loss,Mae
1,0.097666,0.098243,2.683522
2,0.095676,0.095610,2.580921
3,0.093661,0.095393,2.612401


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## **Extra: Data Augmentation & Upsampling**
To improve the model's performance on specific regions or edge cases, we use an **Upsampling strategy**. This function merges our primary dataset with a targeted secondary dataset (e.g., new labeled data) and replicates the secondary data to ensure the model gives it sufficient weight during training.

**Key Features:**
* **Balancing:** Prevents the model from being biased toward the larger dataset.
* **Shuffle:** Randomizes the merged data so the model doesn't learn based on the order of sources.

In [15]:
import json
import random

def merge_and_upsample_datasets(path_1, path_2, upsample_factor=3):
    """
    Merges two JSON datasets and applies upsampling to the second dataset.

    :param path_1: Path to the primary/larger dataset.
    :param path_2: Path to the targeted dataset for upsampling.
    :param upsample_factor: The multiplier for duplicating the second dataset.
    :return: A combined and shuffled list of data entries.
    """

    # 1. Load the datasets from JSON files
    with open(path_1, 'r', encoding='utf-8') as f:
        # Loading a subset (3000 entries) for balanced distribution
        data_1 = json.load(f)[0:3000]

    with open(path_2, 'r', encoding='utf-8') as f:
        data_2 = json.load(f)

    print(f"Dataset 1 original size: {len(data_1)}")
    print(f"Dataset 2 original size: {len(data_2)}")

    # 2. Perform Upsampling on the second dataset
    # We duplicate the list to give the model more exposure to these specific examples
    upsampled_data_2 = data_2 * upsample_factor
    print(f"Dataset 2 after upsampling (x{upsample_factor}): {len(upsampled_data_2)}")

    # 3. Combine the two datasets into one
    combined_data = data_1 + upsampled_data_2

    # 4. Shuffle the data
    # Critical for Machine Learning: prevents the model from learning the source order
    random.shuffle(combined_data)

    print(f"Total combined dataset size: {len(combined_data)}")
    return combined_data

if __name__ == "__main__":
    # Define file paths
    primary_data_path = "reviews_with_rates_10k.json"
    new_labeled_data_path = "Thailand_labeled.json"

    # Merge datasets with a 3x upsample factor for the new data
    final_dataset = merge_and_upsample_datasets(primary_data_path, new_labeled_data_path, upsample_factor=3)

    # Save the final augmented dataset for the training process
    output_path = "for_full_fineTune.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_dataset, f, indent=2, ensure_ascii=False)

    print(f"Processing complete! File saved as: {output_path}")

Dataset 1 size: 3000
Dataset 2 original size: 922
Dataset 2 after upsampling (x3): 2766
Total combined size: 5766
Final dataset is ready for training!


## **0. Data Access & Storage**
We mount Google Drive to access our datasets and save model checkpoints permanently. This ensures that our progress is not lost when the Colab session expires.

In [6]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths to your data inside Drive for easy access
# Replace this with your actual folder path
DRIVE_PATH = "/content/drive/MyDrive/Your_Project_Folder"

if os.path.exists(DRIVE_PATH):
    print("✅ Successfully connected to project folder.")
else:
    print("⚠️ Drive mounted, but project folder not found. Please check the path.")

Mounted at /content/drive


In [7]:
# יצירת תיקייה מקומית זמנית בתוך הקולאב
!mkdir -p /content/local_model

# העתקת המודל מהדרייב לתיקייה המקומית (זה ייקח דקה-שתיים, אבל יאיץ את האימון פי כמה)
!cp -r /content/drive/MyDrive/checkpoint-7515/* /content/local_model/

# עכשיו בתוך הקוד שלך, שנה את הכתובת ל:
model_path = "/content/local_model"

## **1.1 Hardware Acceleration Check**
Training Transformer models like **RoBERTa** or **DeBERTa** is computationally expensive. To perform the thousands of matrix multiplications required for each training step, we must use a **GPU** (Graphics Processing Unit).

Running this on a standard CPU would be significantly slower (often 10x to 50x slower). This section verifies that the environment is correctly configured to use NVIDIA's CUDA cores.

In [1]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU is NOT active. Training will be very slow.")

✅ GPU is active: Tesla T4


## **9. Model Inference: Testing on Real Data**
Now that our model is trained and refined, we can test its performance on unseen travel descriptions. This section demonstrates how to load a saved checkpoint and use it to predict scores for a new review.

**The Inference Process:**
1. **Model Loading:** We load the fine-tuned weights and the specialized tokenizer.
2. **Preprocessing:** The raw input text is converted into tensors that the model can understand.
3. **Evaluation Mode:** We switch the model to `eval()` mode to disable dropout layers for consistent results.
4. **Rescaling:** Since the model predicts values in a 0-1 range, we multiply the output by 10 to return it to our original human-readable scale.

In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def load_and_predict(text, model_path):
    """
    Loads a fine-tuned model and performs inference on a single review string.

    :param text: The raw text description of a tourist attraction.
    :param model_path: The local path to the model checkpoint.
    :return: A dictionary mapping aspect labels to predicted scores (0-10).
    """
    # 1. Load tokenizer and model
    # We use the base model name for the tokenizer if not saved locally
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # Set model to evaluation mode (disables dropout)
    model.eval()

    # 2. Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # 3. Perform the forward pass (Inference)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 4. Post-process the output
    # Convert from 0-1 normalized range back to 0-10
    predictions = logits * 10.0

    # Define the aspect keys in the same order as during training
    label_keys = ["Romance", "Family", "Cost", "Nature", "Adventure",
                  "Culture", "Food", "Relaxation", "Service", "Accessibility"]

    # Map scores to labels
    results = dict(zip(label_keys, predictions[0].tolist()))
    return results

# --- Testing the Inference ---

# Define the path to your best checkpoint
checkpoint_path = "/content/tourism_model/checkpoint-1625"

sample_review = """
Our itinerary begins in Asakusa, home to one of the most impressive temple complexes in Tokyo.
You can easily reach this area with the Asakusa metro line.
"""

# Run prediction
print("Running inference on sample review...")
prediction = load_and_predict(sample_review, checkpoint_path)

print("\n--- Predictions (Scale 0-10) ---")
for label, score in prediction.items():
    # We round the score for better readability
    print(f"{label:15}: {round(score, 2)}")

Running inference on sample review...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


--- Predictions (Scale 0-10) ---
Romance        : 3.02
Family         : 4.74
Cost           : 2.33
Nature         : 6.05
Adventure      : 4.91
Culture        : 10.03
Food           : 0.66
Relaxation     : 7.29
Service        : 6.4
Accessibility  : 8.79


## **8. Exporting the Model**
After the fine-tuning process is complete, we need to export the model for deployment or local testing. Since a Transformer model consists of multiple files (weights, configuration, tokenizer vocabulary), we compress the entire directory into a **ZIP archive** for easy downloading.

In [22]:
import shutil
from google.colab import files

# 1. Define paths
# source_dir should point to the specific checkpoint or final model folder
source_dir = '/content/tourism_model/checkpoint-1625'
output_filename = 'tourism_model_checkpoint_1625' # The name of the resulting ZIP file

# 2. Create the ZIP archive
# shutil.make_archive handles the compression of the entire directory
shutil.make_archive(output_filename, 'zip', source_dir)

print(f"✅ ZIP file created successfully: {output_filename}.zip")

# 3. Trigger automatic download to your local machine
# This will prompt your browser to save the file
files.download(f"{output_filename}.zip")

✅ ZIP file created: tourism_model_checkpoint_1625.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>